[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/33_computer_vision/02_transfer_learning_and_fine_tuning.ipynb)

# Bài 02 — Transfer learning và fine-tuning

**🎯 Mục tiêu**
- Ý tưởng **học chuyển giao**: tái sử dụng đặc trưng học từ một bài toán lớn (nguồn) cho bài toán mới ít nhãn (đích).
- Ba chiến lược: train **từ đầu**, **linear probe** (đóng băng backbone), **fine-tune** (cập nhật cả backbone, learning rate nhỏ hơn).
- Thí nghiệm theo số nhãn: chiến lược nào thắng khi có 20, 80, 400 ảnh.
- Các chi tiết thực hành: BatchNorm, learning rate phân tầng, tiền xử lý khớp mô hình gốc; dùng mô hình tiền huấn luyện thật (torchvision).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/33_computer_vision

In [ ]:
import copy
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

from ml_course.datasets import SHAPE_CLASSES, SHAPE_CLASSES_EXTRA, make_shapes
from ml_course.visualization import set_style

try:
    import torch
    import torch.nn.functional as F
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
to_tensor = lambda X: torch.tensor(X[:, None], dtype=torch.float32)
print("thiết bị:", device)

## 1. Ý tưởng

Mạng train trên một bài toán lớn học được **phân cấp đặc trưng**: lớp đầu phát hiện cạnh, góc, kết cấu — gần như **chung cho mọi ảnh**; lớp sau tổ hợp thành bộ phận, hình dạng — càng về sau càng đặc thù cho bài toán nguồn. Với bài toán mới ít nhãn, thay vì học lại mọi thứ từ vài chục ảnh, ta **mượn** phần lớn mạng đã train (backbone) và chỉ học thêm phần nhỏ.

Trong thực tế, nguồn là ImageNet (1.2 triệu ảnh, 1 000 lớp) hoặc các mô hình nền tảng (CLIP, DINOv2) train trên hàng trăm triệu ảnh. Ở đây mô phỏng ở quy mô nhỏ, chạy offline:
- **Nguồn**: 30 000 ảnh, 5 hình cũ (tròn, vuông, tam giác, chữ thập, vành khuyên).
- **Đích**: 4 hình **mới** (ngôi sao, lục giác, bán nguyệt, chữ L) — chỉ có 20 / 80 / 400 ảnh có nhãn.

In [ ]:
X_target_pool, y_target_pool = make_shapes(2000, seed=20, classes=SHAPE_CLASSES_EXTRA)
X_target_test, y_target_test = make_shapes(2000, seed=21, classes=SHAPE_CLASSES_EXTRA)
X_target_pool, X_target_test = to_tensor(X_target_pool), to_tensor(X_target_test)
y_target_pool, y_target_test = torch.tensor(y_target_pool), torch.tensor(y_target_test)

X_src_show, y_src_show = make_shapes(5, seed=1)
fig, axes = plt.subplots(2, 5, figsize=(11, 4.4))
for j in range(5):
    axes[0, j].imshow(X_src_show[j], cmap="gray")
    axes[0, j].set_title(f"nguồn: {SHAPE_CLASSES[y_src_show[j]]}", fontsize=9)
    axes[1, j].imshow(X_target_pool[j, 0], cmap="gray")
    axes[1, j].set_title(f"đích: {SHAPE_CLASSES_EXTRA[y_target_pool[j]]}", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Tiền huấn luyện backbone trên bài toán nguồn

Kiến trúc: 4 lớp conv + BatchNorm + global average pooling → vector đặc trưng 128 chiều (backbone), cộng một lớp tuyến tính (head) theo số lớp.

In [ ]:
class Net(nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        def block(i, o):
            return [nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU()]

        self.backbone = nn.Sequential(*block(1, 32), nn.MaxPool2d(2), *block(32, 64), nn.MaxPool2d(2),
                                      *block(64, 128), *block(128, 128), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.head = nn.Linear(128, n_classes)

    def forward(self, x):
        return self.head(self.backbone(x))


def train(model, X, y, steps, param_groups=None, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    model.to(device)
    opt = torch.optim.Adam(param_groups if param_groups is not None else model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        model.train()
        idx = torch.randint(0, len(X), (64,), generator=g)
        opt.zero_grad()
        F.cross_entropy(model(X[idx].to(device)), y[idx].to(device)).backward()
        opt.step()
    return model


@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    return sum((model(X[i:i + 500].to(device)).argmax(1).cpu() == y[i:i + 500]).sum().item() for i in range(0, len(X), 500)) / len(X)


@torch.no_grad()
def features(model, X):
    model.eval()
    return torch.cat([model.backbone(X[i:i + 500].to(device)).cpu() for i in range(0, len(X), 500)]).numpy()


X_src, y_src = make_shapes(30000, seed=10)
X_src_test, y_src_test = make_shapes(1000, seed=11)
torch.manual_seed(0)
t0 = time.perf_counter()
source_model = train(Net(5), to_tensor(X_src), torch.tensor(y_src), steps=3000)
print(f"mô hình nguồn: accuracy test (5 hình cũ) = {accuracy(source_model, to_tensor(X_src_test), torch.tensor(y_src_test)):.3f} | "
      f"{time.perf_counter() - t0:.0f}s")

## 3. Đặc trưng có chuyển giao được không?

Đưa ảnh của 4 hình **mới** (mô hình nguồn chưa từng thấy) qua backbone, chiếu PCA 2D — so với backbone khởi tạo ngẫu nhiên:

In [ ]:
torch.manual_seed(0)
random_model = Net(5).to(device)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (name, m) in zip(axes, [("backbone ngẫu nhiên", random_model), ("backbone tiền huấn luyện", source_model)]):
    Z = PCA(2).fit_transform(features(m, X_target_test[:800]))
    for c, cname in enumerate(SHAPE_CLASSES_EXTRA):
        sel = (y_target_test[:800] == c).numpy()
        ax.scatter(Z[sel, 0], Z[sel, 1], s=8, label=cname, alpha=0.6)
    ax.set_title(name)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Không được train trên 4 hình mới, nhưng backbone tiền huấn luyện đã tách chúng thành các cụm khá rõ — các đặc trưng "góc nhọn", "cạnh cong", "lỗ ở giữa"... học từ 5 hình cũ vẫn hữu ích cho hình mới. Backbone ngẫu nhiên trộn lẫn các lớp.

## 4. Ba chiến lược khi ít nhãn

- **Từ đầu**: khởi tạo ngẫu nhiên, train toàn bộ trên dữ liệu đích.
- **Linear probe**: đóng băng backbone, chỉ học bộ phân loại tuyến tính trên đặc trưng (ở đây: `LogisticRegression` trên vector 128 chiều). Ít tham số nhất → ít overfit nhất.
- **Fine-tune**: thay head mới, train **cả** mạng với learning rate **phân tầng** — backbone nhỏ (1e-4) để không phá đặc trưng tốt, head lớn hơn (1e-3). Biến thể "**head trước**": train riêng head vài trăm bước (backbone đóng băng) rồi mới mở backbone — tránh gradient lớn từ head ngẫu nhiên làm hỏng backbone ở những bước đầu.

Mỗi cấu hình: 600 bước cập nhật, 3 lần chọn mẫu khác nhau; test trên 2 000 ảnh.

In [ ]:
def fine_tune(n_classes, X, y, seed, head_first=False):
    model = copy.deepcopy(source_model)
    torch.manual_seed(seed)
    model.head = nn.Linear(128, n_classes)
    model.to(device)
    steps = 600
    if head_first:                                                      # 200 bước chỉ head, sau đó cả mạng
        for p in model.backbone.parameters():
            p.requires_grad_(False)
        train(model, X, y, steps=200, param_groups=model.head.parameters(), seed=seed)
        for p in model.backbone.parameters():
            p.requires_grad_(True)
        steps = 400
    groups = [{"params": model.backbone.parameters(), "lr": 1e-4}, {"params": model.head.parameters(), "lr": 1e-3}]
    return train(model, X, y, steps=steps, param_groups=groups, seed=seed)


def run_strategies(n, seeds=range(3)):
    res = {"từ đầu": [], "linear probe": [], "fine-tune": [], "fine-tune (head trước)": []}
    for seed in seeds:
        idx = torch.randperm(len(X_target_pool), generator=torch.Generator().manual_seed(seed))[:n]
        X, y = X_target_pool[idx], y_target_pool[idx]
        torch.manual_seed(seed)
        res["từ đầu"].append(accuracy(train(Net(4), X, y, steps=600, seed=seed), X_target_test, y_target_test))
        probe = LogisticRegression(max_iter=3000).fit(features(source_model, X), y.numpy())
        res["linear probe"].append((probe.predict(features(source_model, X_target_test)) == y_target_test.numpy()).mean())
        res["fine-tune"].append(accuracy(fine_tune(4, X, y, seed), X_target_test, y_target_test))
        res["fine-tune (head trước)"].append(accuracy(fine_tune(4, X, y, seed, head_first=True), X_target_test, y_target_test))
    return {k: np.mean(v) for k, v in res.items()}


transfer = {}
for n in [20, 80, 400]:
    t0 = time.perf_counter()
    transfer[n] = run_strategies(n)
    print(f"{n:3d} ảnh có nhãn: " + " | ".join(f"{k} = {v:.3f}" for k, v in transfer[n].items()) + f" | {time.perf_counter() - t0:.0f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ns = list(transfer)
for k in transfer[ns[0]]:
    ax.plot(ns, [transfer[n][k] for n in ns], "o-", label=k)
ax.set_xscale("log")
ax.set_xticks(ns, [str(n) for n in ns])
ax.set_xlabel("số ảnh có nhãn của bài toán đích")
ax.set_ylabel("accuracy test")
ax.legend()
plt.tight_layout()
plt.show()

- **Rất ít nhãn (20 ảnh, 5 ảnh/lớp)**: train từ đầu gần như vô dụng; **linear probe** (chỉ học 4 × 129 tham số, khó overfit) và **fine-tune có train head trước** tốt nhất, ngang nhau; fine-tune thẳng kém hơn một chút — ở những bước đầu, gradient từ head khởi tạo ngẫu nhiên làm xáo trộn backbone.
- **Trung bình (80)**: các cách dùng backbone tiền huấn luyện đều vượt xa từ đầu; fine-tune bắt đầu ngang/vượt linear probe.
- **Nhiều hơn (400)**: **fine-tune** thắng rõ linear probe — đủ dữ liệu để **điều chỉnh** đặc trưng cho các hình mới; train từ đầu vẫn kém dù đã có 100 ảnh/lớp.

Quy tắc thực hành: ít dữ liệu và miền gần nguồn → linear probe / chỉ train lớp cuối; nhiều dữ liệu hơn hoặc miền khác xa → fine-tune (có thể mở dần từ lớp cuối về đầu); gần như **không bao giờ** nên train từ đầu khi có mô hình tiền huấn luyện phù hợp.

## 5. Chi tiết thực hành

- **Tiền xử lý khớp mô hình gốc**: cùng kích thước đầu vào, cùng chuẩn hoá (ImageNet: mean `[0.485, 0.456, 0.406]`, std `[0.229, 0.224, 0.225]`) — torchvision cung cấp sẵn qua `weights.transforms()`.
- **BatchNorm**: khi batch nhỏ hoặc dữ liệu đích rất ít, có thể giữ BN ở chế độ `eval` (dùng thống kê của nguồn) hoặc đóng băng tham số BN.
- **Learning rate**: nhỏ hơn 10–100 lần so với train từ đầu cho backbone; warmup ngắn; ít epoch.
- **Catastrophic forgetting**: fine-tune làm mô hình quên bài toán nguồn — không sao nếu chỉ cần bài toán đích.
- **Miền khác xa** (ảnh y tế, vệ tinh, ảnh hiển vi): đặc trưng tầng thấp vẫn có ích; cân nhắc mô hình tiền huấn luyện trên miền gần hơn hoặc self-supervised trên dữ liệu không nhãn của chính mình (chương 34).
- **Mô hình nền tảng** (CLIP, DINOv2, SigLIP): đặc trưng tốt đến mức linear probe hoặc thậm chí **zero-shot** (phân loại bằng mô tả văn bản, không cần ảnh có nhãn) thường đã đủ.

Đoạn code tham khảo với mô hình ImageNet thật (cần mạng để tải trọng số ~45 MB; mặc định không chạy):

In [ ]:
RUN_PRETRAINED = False                                                 # đổi thành True nếu có mạng

if RUN_PRETRAINED:
    from torchvision.models import ResNet18_Weights, resnet18

    weights = ResNet18_Weights.DEFAULT
    model = resnet18(weights=weights)
    preprocess = weights.transforms()                                  # resize 256 → crop 224 → chuẩn hoá ImageNet
    for p in model.parameters():                                       # linear probe: đóng băng toàn bộ
        p.requires_grad_(False)
    model.fc = nn.Linear(model.fc.in_features, len(SHAPE_CLASSES_EXTRA))   # head mới, mặc định requires_grad=True
    x = X_target_pool[:8].repeat(1, 3, 1, 1)                           # ảnh xám → 3 kênh
    print("đầu ra:", tuple(model(preprocess(x)).shape))
    # fine-tune: mở khoá dần, vd model.layer4 trước; optimizer với param groups lr khác nhau như mục 4
else:
    print("Bỏ qua phần mô hình tiền huấn luyện thật (RUN_PRETRAINED = False).")

## 🧠 Kiểm tra nhanh

1. Vì sao đặc trưng học từ bài toán nguồn dùng được cho bài toán đích khác lớp?
2. Linear probe và fine-tune khác nhau thế nào? Khi nào chọn mỗi cách?
3. Vì sao fine-tune dùng learning rate nhỏ cho backbone? "Head trước" giải quyết vấn đề gì?
4. Những gì phải khớp với mô hình gốc khi dùng mô hình tiền huấn luyện?
5. Khi miền đích khác xa miền nguồn, nên làm gì?

## 📌 Tóm tắt

- Transfer learning tái sử dụng phân cấp đặc trưng; với ít nhãn, nó vượt xa train từ đầu.
- Rất ít nhãn → linear probe; nhiều nhãn hơn → fine-tune với LR phân tầng (có thể train head trước).
- Khớp tiền xử lý, cẩn thận BatchNorm, LR nhỏ; mô hình nền tảng cho đặc trưng mạnh tới mức linear probe/zero-shot thường đủ.

## 📚 Tài liệu tham khảo

- Yosinski et al. (2014) — *How transferable are features in deep neural networks?*
- Kornblith, Shlens & Le (2019) — *Do Better ImageNet Models Transfer Better?*
- Howard & Ruder (2018) — *Universal Language Model Fine-tuning* (discriminative LR, gradual unfreezing)
- Kumar et al. (2022) — *Fine-Tuning can Distort Pretrained Features and Underperform Out-of-Distribution* (LP-FT)
- Radford et al. (2021) — *CLIP*; Oquab et al. (2023) — *DINOv2*
- [PyTorch — Transfer Learning for Computer Vision Tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)